# NiyamTrace-X Q1 Wave 4 / Experiment C
## Verified Execution Broker V3 — Concurrency, Replay, Crash Safety and Latency

**Recommended:** run in Colab Account C. CPU only.

This strengthens the current broker evidence by testing:
- actor/tool/argument/contract/policy/snapshot/effect-set/nonce/expiry binding;
- exact commit delta;
- replay across broker restart;
- stale state;
- forged capability and target tampering;
- injected crash rollback;
- phantom-writer concurrency;
- double-submit concurrency;
- direct-vs-broker p50/p95/p99 latency;
- zero-event upper bounds.

**Claim boundary:** this remains a reference SQLite broker. It is not production-complete mediation
unless the real production executor is forced through this path.

In [ ]:
# CELL 1 — Setup
from pathlib import Path
import sqlite3, json, hashlib, hmac, secrets, time, threading, tempfile, random, os, sys, math, zipfile, platform
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED=20261004
random.seed(SEED); np.random.seed(SEED)

BASE=Path("/content") if Path("/content").exists() else Path("/mnt/data")
ROOT=BASE/"NTX_Q1_W4_C_BROKER_V3"
RESULTS,FIGURES,RAW=ROOT/"results",ROOT/"figures",ROOT/"raw"
for d in [RESULTS,FIGURES,RAW]: d.mkdir(parents=True,exist_ok=True)

MODE=os.getenv("NTX_MODE","PAPER").upper()
assert MODE in {"SMOKE","PAPER","FULL"}
TRIALS={"SMOKE":10,"PAPER":120,"FULL":250}[MODE]
RACE_REPS={"SMOKE":5,"PAPER":40,"FULL":100}[MODE]
LAT_REPS={"SMOKE":50,"PAPER":500,"FULL":1500}[MODE]
print({"MODE":MODE,"TRIALS":TRIALS,"RACE_REPS":RACE_REPS,"LAT_REPS":LAT_REPS})

In [ ]:
# CELL 2 — DB and cryptographic helpers
SECRET=b"ntx-q1-wave4-broker-v3-secret"
POLICY={"name":"archive-policy","version":3,"allow_tools":["invoice.archive"],"max_rows":25}

def cj(x): return json.dumps(x,sort_keys=True,separators=(",",":"),ensure_ascii=False,default=str)
def hobj(x): return hashlib.sha256(cj(x).encode()).hexdigest()
def phash(p=POLICY): return hobj(p)
def sign(body): return hmac.new(SECRET,cj(body).encode(),hashlib.sha256).hexdigest()

def connect(path):
    c=sqlite3.connect(path,timeout=10,isolation_level=None,check_same_thread=False)
    c.execute("PRAGMA journal_mode=WAL")
    c.execute("PRAGMA busy_timeout=10000")
    c.execute("PRAGMA synchronous=NORMAL")
    return c

def init_db(path,rows=None):
    p=Path(path)
    if p.exists(): p.unlink()
    c=connect(p)
    c.executescript("""
    CREATE TABLE invoices(id TEXT PRIMARY KEY,vendor TEXT,month INTEGER,year INTEGER,status TEXT);
    CREATE TABLE used_nonces(nonce TEXT PRIMARY KEY,used_at REAL NOT NULL,decision_id TEXT NOT NULL);
    CREATE TABLE audit_log(id INTEGER PRIMARY KEY AUTOINCREMENT,decision_id TEXT,event TEXT,detail TEXT,ts REAL);
    """)
    rows=rows or [
        ("I1","7184",11,2025,"OPEN"),("I2","7184",11,2025,"OPEN"),
        ("I3","7184",11,2025,"OPEN"),("I4","9001",11,2025,"OPEN"),
        ("I5","7184",12,2025,"OPEN"),
    ]
    c.executemany("INSERT INTO invoices VALUES(?,?,?,?,?)",rows)
    c.close()

def snapshot_rows(c):
    return c.execute("SELECT id,vendor,month,year,status FROM invoices ORDER BY id").fetchall()
def snapshot_hash(c): return hobj(snapshot_rows(c))
def select_ids(c,args):
    q='SELECT id FROM invoices WHERE vendor=? AND month=? AND year=? AND status="OPEN" ORDER BY id'
    return [r[0] for r in c.execute(q,(str(args["vendor"]),int(args["month"]),int(args["year"])))]

def fresh_db(rows=None):
    p=Path(tempfile.mktemp(prefix="ntxw4_",suffix=".sqlite"))
    init_db(p,rows); return p

ARGS={"vendor":"7184","month":11,"year":2025}
CONTRACT={"action":"invoice.archive","vendor_id":"7184","month":11,"year":2025,"scope":"matching"}

In [ ]:
# CELL 3 — Capability issuance
def issue_capability(path,actor,args,contract,ttl=60,policy=POLICY):
    c=connect(path)
    ids=select_ids(c,args); snap=snapshot_hash(c); c.close()
    body={
        "decision_id":secrets.token_hex(12),"actor":actor,"tool":"invoice.archive",
        "args_hash":hobj(args),"contract_hash":hobj(contract),"policy_hash":phash(policy),
        "snapshot_hash":snap,"predicted_ids":ids,"predicted_ids_hash":hobj(ids),
        "expires_at":time.time()+ttl,"nonce":secrets.token_hex(16),
    }
    return {**body,"signature":sign(body)}

In [ ]:
# CELL 4 — Broker V3
class BrokerV3:
    def __init__(self,path): self.path=Path(path)

    def _static(self,cap,actor,requested_tool,args,contract,policy):
        body={k:v for k,v in cap.items() if k!="signature"}
        if not hmac.compare_digest(sign(body),cap.get("signature","")): return False,"SIGNATURE"
        if time.time()>cap["expires_at"]: return False,"EXPIRED"
        if actor!=cap["actor"]: return False,"ACTOR"
        if requested_tool!=cap["tool"] or requested_tool!="invoice.archive": return False,"TOOL"
        if hobj(args)!=cap["args_hash"]: return False,"ARGS"
        if hobj(contract)!=cap["contract_hash"]: return False,"CONTRACT"
        if phash(policy)!=cap["policy_hash"]: return False,"POLICY"
        if hobj(cap["predicted_ids"])!=cap["predicted_ids_hash"]: return False,"PREDICTED_SET_TAMPER"
        if len(cap["predicted_ids"])>int(policy.get("max_rows",10**9)): return False,"ROW_LIMIT"
        return True,"OK"

    def execute(self,cap,actor,args,contract,policy=POLICY,failpoint=None,inside_tx_hook=None,requested_tool="invoice.archive"):
        c=connect(self.path); t0=time.perf_counter_ns()
        try:
            c.execute("BEGIN IMMEDIATE")
            ok,reason=self._static(cap,actor,requested_tool,args,contract,policy)
            if not ok:
                c.rollback(); return {"committed":False,"reason":reason,"actual_ids":[],"latency_ns":time.perf_counter_ns()-t0}
            if c.execute("SELECT 1 FROM used_nonces WHERE nonce=?",(cap["nonce"],)).fetchone():
                c.rollback(); return {"committed":False,"reason":"REPLAY","actual_ids":[],"latency_ns":time.perf_counter_ns()-t0}
            if snapshot_hash(c)!=cap["snapshot_hash"]:
                c.rollback(); return {"committed":False,"reason":"STALE_SNAPSHOT","actual_ids":[],"latency_ns":time.perf_counter_ns()-t0}

            current=select_ids(c,args)
            if current!=cap["predicted_ids"] or hobj(current)!=cap["predicted_ids_hash"]:
                c.rollback(); return {"committed":False,"reason":"EFFECT_SET_MISMATCH","actual_ids":[],"latency_ns":time.perf_counter_ns()-t0}

            if inside_tx_hook: inside_tx_hook()

            before={r[0]:r[-1] for r in snapshot_rows(c)}
            for i in current:
                c.execute('UPDATE invoices SET status="ARCHIVED" WHERE id=? AND status="OPEN"',(i,))
            after={r[0]:r[-1] for r in snapshot_rows(c)}
            actual=sorted(i for i in before if before[i]!=after[i])
            if sorted(actual)!=sorted(cap["predicted_ids"]):
                c.rollback(); return {"committed":False,"reason":"COMMIT_DELTA_MISMATCH","actual_ids":actual,"latency_ns":time.perf_counter_ns()-t0}

            if failpoint=="after_mutation": raise RuntimeError("injected crash")
            c.execute("INSERT INTO used_nonces(nonce,used_at,decision_id) VALUES(?,?,?)",
                      (cap["nonce"],time.time(),cap["decision_id"]))
            c.execute("INSERT INTO audit_log(decision_id,event,detail,ts) VALUES(?,?,?,?)",
                      (cap["decision_id"],"COMMIT",cj({"ids":actual}),time.time()))
            if failpoint=="before_commit": raise RuntimeError("injected crash")
            c.commit()
            return {"committed":True,"reason":"OK","actual_ids":actual,"latency_ns":time.perf_counter_ns()-t0}
        except Exception as e:
            try:c.rollback()
            except:pass
            return {"committed":False,"reason":"ROLLBACK:"+type(e).__name__,"actual_ids":[],"latency_ns":time.perf_counter_ns()-t0}
        finally:
            c.close()

print("Broker V3 ready.")

In [ ]:
# CELL 5 — Sequential randomized scenario matrix
SCENARIOS=[
    "valid","forged_signature","actor_mismatch","tool_mismatch","args_tamper",
    "contract_tamper","policy_change","issuer_effect_mismatch","replay",
    "stale_snapshot","expired","crash_after_mutation","crash_before_commit"
]
rng=random.Random(SEED)
rows=[]

for scen in SCENARIOS:
    for rep in range(TRIALS):
        dbrows=[
            ("I1","7184",11,2025,"OPEN"),("I2","7184",11,2025,"OPEN"),
            ("I3","7184",11,2025,"OPEN"),("I4",str(8000+rng.randrange(1000)),11,2025,"OPEN")
        ]
        p=fresh_db(dbrows)
        args=dict(ARGS); contract=dict(CONTRACT); policy=dict(POLICY)
        cap=issue_capability(p,"alice",args,contract)
        actor="alice"; failpoint=None; requested_tool="invoice.archive"

        if scen=="forged_signature": cap=dict(cap); cap["signature"]="00"*32
        elif scen=="actor_mismatch": actor="mallory"
        elif scen=="tool_mismatch": requested_tool="invoice.pay"
        elif scen=="args_tamper": args={**args,"vendor":"9001"}
        elif scen=="contract_tamper": contract={**contract,"scope":"all"}
        elif scen=="policy_change": policy={**policy,"version":999}
        elif scen=="issuer_effect_mismatch":
            # Simulate a buggy issuer that signs a wrong predicted set; broker must reject it against current state.
            body={k:v for k,v in cap.items() if k!="signature"}
            body["predicted_ids"]=body["predicted_ids"]+["I999"]
            body["predicted_ids_hash"]=hobj(body["predicted_ids"])
            cap={**body,"signature":sign(body)}
        elif scen=="stale_snapshot":
            c=connect(p); c.execute('INSERT INTO invoices VALUES(?,?,?,?,?)',("IX","7184",11,2025,"OPEN")); c.close()
        elif scen=="expired": cap=issue_capability(p,"alice",args,contract,ttl=-1)
        elif scen=="crash_after_mutation": failpoint="after_mutation"
        elif scen=="crash_before_commit": failpoint="before_commit"

        broker=BrokerV3(p)
        if scen=="replay":
            broker.execute(cap,actor,args,contract,policy,requested_tool=requested_tool)
            out=broker.execute(cap,actor,args,contract,policy,requested_tool=requested_tool)
        else:
            out=broker.execute(cap,actor,args,contract,policy,failpoint=failpoint,requested_tool=requested_tool)

        expected=(scen=="valid")
        rows.append({
            "scenario":scen,"rep":rep,"expected_commit":expected,"committed":out["committed"],
            "reason":out["reason"],"correct":out["committed"]==expected,
            "unsafe_commit":bool(out["committed"] and not expected),
            "false_reject":bool((not out["committed"]) and expected),
            "latency_us":out["latency_ns"]/1000,
        })
        try:p.unlink()
        except:pass

trials=pd.DataFrame(rows)
trials.to_csv(RAW/"scenario_trials.csv",index=False)
display(trials.groupby("scenario").agg(n=("rep","size"),commit_rate=("committed","mean"),
                                        accuracy=("correct","mean"),unsafe=("unsafe_commit","sum"),
                                        median_us=("latency_us","median")).round(4))

In [ ]:
# CELL 6 — Concurrency: phantom insert and double submit
race_rows=[]
for rep in range(RACE_REPS):
    # Phantom writer
    p=fresh_db(); cap=issue_capability(p,"alice",ARGS,CONTRACT)
    started=threading.Event(); done=threading.Event(); info={}
    def writer():
        started.wait(5); t=time.perf_counter()
        c=connect(p)
        try:
            c.execute('INSERT INTO invoices VALUES(?,?,?,?,?)',("PX"+str(rep),"7184",11,2025,"OPEN"))
        finally:c.close()
        info["wait_s"]=time.perf_counter()-t; done.set()
    def hook():
        started.set(); time.sleep(.03)
    th=threading.Thread(target=writer); th.start()
    out=BrokerV3(p).execute(cap,"alice",ARGS,CONTRACT,inside_tx_hook=hook)
    th.join(5)
    c=connect(p); statuses={r[0]:r[-1] for r in snapshot_rows(c)}; c.close()
    safe=bool(out["committed"] and out["actual_ids"]==cap["predicted_ids"] and statuses.get("PX"+str(rep))=="OPEN")
    race_rows.append({"race":"phantom_insert","rep":rep,"safe":safe,"wait_s":info.get("wait_s",np.nan)})
    try:p.unlink()
    except:pass

    # Same capability submitted concurrently: exactly one commit.
    p=fresh_db(); cap=issue_capability(p,"alice",ARGS,CONTRACT); b=BrokerV3(p)
    with ThreadPoolExecutor(max_workers=2) as ex:
        outs=[f.result() for f in [ex.submit(b.execute,cap,"alice",ARGS,CONTRACT),
                                   ex.submit(b.execute,cap,"alice",ARGS,CONTRACT)]]
    commits=sum(o["committed"] for o in outs)
    race_rows.append({"race":"double_submit","rep":rep,"safe":commits==1,"wait_s":np.nan})
    try:p.unlink()
    except:pass

races=pd.DataFrame(race_rows)
races.to_csv(RAW/"race_trials.csv",index=False)
display(races.groupby("race").agg(n=("rep","size"),safe_rate=("safe","mean")).round(4))
assert races.safe.all(), "Concurrency regression detected."

In [ ]:
# CELL 7 — Direct transaction, pre-issued broker, and issue+commit latency
def direct_archive(path,args):
    c=connect(path); t0=time.perf_counter_ns()
    try:
        c.execute("BEGIN IMMEDIATE")
        ids=select_ids(c,args)
        for i in ids:
            c.execute('UPDATE invoices SET status="ARCHIVED" WHERE id=? AND status="OPEN"',(i,))
        c.commit()
        return time.perf_counter_ns()-t0
    finally:c.close()

lat=[]
for rep in range(LAT_REPS):
    p=fresh_db()
    lat.append({"path":"direct_transaction","latency_us":direct_archive(p,ARGS)/1000})
    p.unlink(missing_ok=True)

    p=fresh_db()
    cap=issue_capability(p,"alice",ARGS,CONTRACT)
    out=BrokerV3(p).execute(cap,"alice",ARGS,CONTRACT)
    lat.append({"path":"broker_v3_preissued","latency_us":out["latency_ns"]/1000})
    p.unlink(missing_ok=True)

    p=fresh_db()
    t0=time.perf_counter_ns()
    cap=issue_capability(p,"alice",ARGS,CONTRACT)
    out=BrokerV3(p).execute(cap,"alice",ARGS,CONTRACT)
    total=(time.perf_counter_ns()-t0)/1000
    lat.append({"path":"broker_v3_issue_plus_commit","latency_us":total})
    p.unlink(missing_ok=True)

lat=pd.DataFrame(lat)
lat.to_csv(RAW/"latency_raw.csv",index=False)
lat_sum=[]
for path,g in lat.groupby("path"):
    lat_sum.append({"path":path,"n":len(g),"p50_us":g.latency_us.quantile(.50),
                    "p95_us":g.latency_us.quantile(.95),"p99_us":g.latency_us.quantile(.99),
                    "mean_us":g.latency_us.mean()})
lat_sum=pd.DataFrame(lat_sum)
lat_sum.to_csv(RESULTS/"latency_summary.csv",index=False)
display(lat_sum.round(2))

In [ ]:
# CELL 8 — Overall metrics and confidence bound
scenario=(trials.groupby("scenario")
          .agg(n=("rep","size"),commit_rate=("committed","mean"),
               accuracy=("correct","mean"),unsafe_commits=("unsafe_commit","sum"),
               false_rejects=("false_reject","sum"))
          .reset_index())
scenario.to_csv(RESULTS/"scenario_summary.csv",index=False)

unsafe=int(trials.unsafe_commit.sum())
opps=int((~trials.expected_commit).sum())
upper=(1-0.05**(1/opps)) if unsafe==0 and opps else np.nan
overall=pd.DataFrame([{
    "trials":len(trials),"valid_trials":int(trials.expected_commit.sum()),
    "invalid_or_fault_trials":opps,"correct_rate":trials.correct.mean(),
    "unsafe_commits":unsafe,"unsafe_commit_rate":unsafe/opps if opps else np.nan,
    "zero_event_upper95":upper,"false_rejects":int(trials.false_reject.sum()),
    "race_trials":len(races),"race_safe_rate":races.safe.mean(),
}])
overall.to_csv(RESULTS/"overall_metrics.csv",index=False)
display(overall)

In [ ]:
# CELL 9 — Figures
fig,ax=plt.subplots(figsize=(8,5))
plot=scenario.sort_values("commit_rate")
ax.barh(plot.scenario,plot.commit_rate)
ax.set_xlim(0,1.02); ax.set_xlabel("Observed commit rate")
ax.set_title("Broker V3 commit behavior across valid and fault scenarios")
ax.grid(axis="x",alpha=.25)
fig.tight_layout(); fig.savefig(FIGURES/"scenario_commit_rates.png",dpi=220); plt.show()

fig,ax=plt.subplots(figsize=(6.4,4.6))
names=list(lat.path.unique())
ax.boxplot([lat[lat.path==p].latency_us.values for p in names],labels=names,showfliers=False)
ax.set_ylabel("Latency (µs)")
ax.set_title("Direct transaction vs pre-issued Broker V3 commit path")
fig.tight_layout(); fig.savefig(FIGURES/"broker_latency.png",dpi=220); plt.show()

In [ ]:
# CELL 10 — Paper summary, manifest and ZIP
o=overall.iloc[0]
lines=[
    "# Verified Execution Broker V3",
    "",
    "- Sequential trials: **"+str(int(o.trials))+"**",
    "- Invalid/fault opportunities: **"+str(int(o.invalid_or_fault_trials))+"**",
    "- Unsafe commits: **"+str(int(o.unsafe_commits))+"**",
    "- Unsafe-commit rate: **"+format(o.unsafe_commit_rate,".3%")+"**",
    "- One-sided 95% zero-event upper bound (when zero events): **"+format(o.zero_event_upper95,".3%")+"**",
    "- Concurrency safe rate: **"+format(o.race_safe_rate,".2%")+"**",
    "",
    "Claim boundary: reference SQLite broker, not production-complete mediation.",
]
(RESULTS/"paper_ready_summary.md").write_text("\n".join(lines))
manifest={
    "experiment":"NTX_Q1_W4_C_Broker_V3","seed":SEED,"mode":MODE,
    "trials_per_scenario":TRIALS,"race_reps":RACE_REPS,"latency_reps":LAT_REPS,
    "scenarios":SCENARIOS,"python":sys.version,"platform":platform.platform(),
}
(RESULTS/"MANIFEST.json").write_text(json.dumps(manifest,indent=2))

zip_path=BASE/"NTX_Q1_W4_C_BROKER_V3_RESULTS.zip"
if zip_path.exists():zip_path.unlink()
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for folder in [RESULTS,FIGURES,RAW]:
        for p in folder.rglob("*"):
            if p.is_file(): z.write(p,p.relative_to(ROOT))
print("Created:",zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass